# DỰ ĐOÁN GIÁ CỔ PHIẾU VN-INDEX
## *Dựa trên biến động giá trong quá khứ — mã thử nghiệm: VCB*

**Môn:** Học Máy — HCMUTE  
**Sinh viên thực hiện:** Bá Hoài Sơn — Bùi Thanh Tú  
**Dữ liệu:** Yahoo Finance qua thư viện `yfinance`

## 1. Bài toán

- Có thể dự đoán **biến động giá ngày kế tiếp** của cổ phiếu chỉ từ giá lịch sử không?
- **Mã thử nghiệm:** VCB (Vietcombank — `VCB.VN`).
- **Loại bài toán:** Hồi quy có giám sát trên chuỗi thời gian.
- **Ý nghĩa:** Hỗ trợ ra quyết định đầu tư, kiểm chứng giả thuyết Hiệu Quả Thị Trường (EMH).

## 2. Dữ liệu

- **Nguồn:** Yahoo Finance qua `yfinance` → lưu `data/vcb_stock.csv`.
- **Quy mô:** 7.673 quan sát (4.209 phiên ngày + 3.464 phiên giờ).
- **Khoảng thời gian:** 2009-06-30 → 2026-05-15.
- **Cột:** `Date`, `Open`, `High`, `Low`, `Close`, `Volume`, `Interval`, `Ticker`, `Symbol`.

## 3. Lịch sử giá VCB

![](../image/price_history_daily.png)

## 4. Cạm bẫy: dự đoán giá tuyệt đối

Khi dùng giá tuyệt đối làm input/output:

| Mô hình | RMSE | R² |
|---------|------|-----|
| Linear Regression | 873 | **0.96** |
| Random Forest | 9.480 | -3.79 |
| KNN | 11.387 | -5.92 |

- LR "thắng" giả tạo — chỉ **sao chép** giá hôm nay.
- Tree/KNN R² âm vì **không thể ngoại suy** ngoài miền giá đã thấy.
- → Phải biến đổi bài toán.

## 5. Giải pháp: dự đoán LỢI SUẤT

$$\hat r_{t+1} = f(\mathbf{x}_t),\qquad \hat C_{t+1} = C_t \cdot (1 + \hat r_{t+1})$$

**Đặc trưng đều STATIONARY:**

- Lợi suất trễ: `Return_{1, 2, 3, 5, 10}`
- Tỷ lệ với MA: `MA{5,10,20}_Ratio`
- Volatility: `Vol_{5, 10}`
- RSI(14)
- Biên độ trong ngày: `HL_Range`, `OC_Range`
- Khối lượng: `Vol_Change`

## 6. Bốn mô hình thử nghiệm

| Mô hình | Vai trò |
|---------|---------|
| **Linear Regression** | Baseline tuyến tính |
| **K-Nearest Neighbors (k=15)** | Phi tham số, học theo các phiên "giống" |
| **Random Forest (400 cây)** | Phi tuyến, robust với nhiễu |
| **Voting Ensemble** | Trung bình dự đoán của 3 mô hình trên |

*Tất cả bao bằng `Pipeline(StandardScaler + estimator)`.*

## 7. Pipeline & cách đánh giá

1. Load dữ liệu (`load_raw_data`).
2. Sinh đặc trưng (`build_features`).
3. Chia **theo trật tự thời gian** 80% train / 20% test (không xáo trộn).
4. Huấn luyện 4 mô hình.
5. Đánh giá: **MAE, RMSE, R²** (lợi suất) + **DirAcc(%)** + **MAPE giá**.
6. Trực quan và kết luận.

In [1]:
import sys
sys.path.insert(0, '../scripts')
from ml_utils import run_full_pipeline

result = run_full_pipeline(interval='1d', train_ratio=0.8, save_images=False)
print('--- METRIC TRÊN MIỀN LỢI SUẤT ---')
display(result['metrics_return'])
print('--- METRIC TRÊN MIỀN GIÁ (VND) ---')
display(result['metrics_price'])

--- METRIC TRÊN MIỀN LỢI SUẤT ---


,MAE,RMSE,R2,DirAcc(%)
Model,,,,
Linear Regression,0.009947,0.014734,-0.008529,43.436754
KNN,0.010691,0.015518,-0.118707,44.152745
Random Forest,0.009812,0.014670,0.000286,44.272076
Ensemble,0.010002,0.014817,-0.019962,43.436754


--- METRIC TRÊN MIỀN GIÁ (VND) ---


,MAE (VND),RMSE (VND),MAPE(%)
Model,,,
Linear Regression,588.78,882.00,0.99
KNN,631.47,924.85,1.07
Random Forest,580.47,878.92,0.98
Ensemble,591.50,885.99,1.00


## 8. So sánh 4 mô hình

![](../image/model_comparison.png)

- **Random Forest** và **Linear Regression** đứng đầu.
- KNN kém nhất (R² âm trên miền lợi suất).
- Ensemble **không** vượt được Random Forest đơn lẻ — bị KNN "kéo xuống".

## 9. Dự đoán vs. Thực tế (giá)

![](../image/predictions_vs_actual.png)

- Cả 4 đường dự đoán **bám rất sát** giá thực tế.
- MAE giá ~ 580–630 VND (**MAPE ~ 1 %**).

## 10. Hiệu Quả Thị Trường — phát hiện chính

- **DirAcc ~ 43–44 %** — không vượt được random 50 %.
- Mô hình dự đoán **mức giá** chính xác chỉ vì quán tính ngắn hạn $\hat C_{t+1} \approx C_t$.
- Nhưng dự đoán **chiều** biến động (lên/xuống) thì gần như ngẫu nhiên.
- → Phù hợp với dạng yếu của Hiệu Quả Thị Trường (EMH): giá lịch sử **không** đủ để kiếm lợi nhuận vượt trội.

## 11. Hạn chế & Hướng phát triển

**Hạn chế:**

- Chỉ dùng giá quá khứ, **không** xét tin tức / yếu tố vĩ mô.
- Không dự đoán được sốc thị trường (COVID, biến động chính sách…).
- Đánh giá theo một lần chia 80/20 — chưa walk-forward.

**Hướng phát triển:**

- Bổ sung sentiment (tin tức, mạng xã hội).
- Mô hình tuần tự (LSTM, Temporal Fusion Transformer).
- Walk-forward validation, thử trên rổ VN30 / VN-Index.
- Tối ưu siêu tham số bằng `TimeSeriesSplit` + grid search.

## 12. Cảm ơn — Q & A

**Bá Hoài Sơn — Bùi Thanh Tú**  
*Học Máy — HCMUTE*

Mã nguồn: `scripts/ml_utils.py`  
Tái lập: `python scripts/fetch_data.py` → mở 4 notebook.